# 2D Curved PN Junction Drift–Diffusion Solver

Run this notebook from `pn_junction_2d/`. It imports the tested Python modules and writes plots and CSV data to `figures/`. The full sweep and grid refinement can take several minutes.

## 1. Governing Equations, Scaling & Discretization

In $\text{cm}$-based semiconductor units at $T = 300\text{ K}$, steady-state transport is governed by:
1. **Poisson:** $\nabla^2 \psi = -\frac{q}{\varepsilon}\big(p - n + N_{\text{net}}(x,y)\big)$
2. **Continuity:** $\nabla \cdot \mathbf{J}_n = q R$, $\qquad \nabla \cdot \mathbf{J}_p = -q R$
3. **Drift-Diffusion:** $\mathbf{J}_n = -q\mu_n n \nabla\psi + q D_n \nabla n$, $\qquad \mathbf{J}_p = -q\mu_p p \nabla\psi - q D_p \nabla p$, with $D_{n,p} = \mu_{n,p} V_T$.
4. **SRH Recombination:** $R(n,p) = \frac{np - n_i^2}{\tau_p(n + n_i) + \tau_n(p + n_i)}$.

Scaling by $V_T = k_B T / q$, $C_0 = \max(N_A, N_D)$, $L_D = \sqrt{\varepsilon V_T / (q C_0)}$, $\mu_0 = \mu_n$, $t_0 = L_D^2/(\mu_0 V_T)$, and $J_0 = q C_0 \mu_0 V_T / L_D$ gives dimensionless equations:
$$\tilde{\nabla}^2 \tilde{\psi} = \tilde{n} - \tilde{p} - \tilde{N}_{\text{net}}, \qquad -\tilde{\nabla}\cdot\tilde{\mathbf{J}}_n = -\tilde{R}, \qquad +\tilde{\nabla}\cdot\tilde{\mathbf{J}}_p = -\tilde{R}.$$

Across each face $x_{i+1/2,j}$ with $z = \tilde{\psi}_{i+1,j} - \tilde{\psi}_{i,j}$ and $B(z) = z/(e^z - 1)$:
$$\tilde{J}_{n,i+1/2,j} = \frac{\tilde{\mu}_n}{\Delta\tilde{x}}\big[B(z)\tilde{n}_{i+1,j} - B(-z)\tilde{n}_{i,j}\big], \qquad \tilde{J}_{p,i+1/2,j} = -\frac{\tilde{\mu}_p}{\Delta\tilde{x}}\big[B(-z)\tilde{p}_{i+1,j} - B(z)\tilde{p}_{i,j}\big].$$

## 2. Scaling, Bernoulli Function, FVM Operators & Junction Geometry

Notice two key numerical details:
- **Neumann boundary control volumes:** Nodes on $j = 0$ and $j = N_y - 1$ own half-height control volumes ($\Delta y / 2$), so omitting the outer face flux divides the inner $y$-flux by $\Delta y / 2$. Terminal current integration uses matching trapezoid weights $w_j$ ($\Delta y / 2$ at the walls, $\Delta y$ in the interior).
- **Smooth junction curve:** Using $x = f(y)$ with $f'(0) = f'(L_y) = 0$ (such as a cosine interface) avoids symmetry kinks at the Neumann walls, and $w \ge 2.5 \max(\Delta x, \Delta y)\sqrt{1 + \max |f'|^2}$ resolves the normal transition width.

## 3. Equilibrium Newton-Raphson, Scharfetter-Gummel Continuity & Gummel Solver

- **Equilibrium ($V_{\text{app}}=0$):** Solves $L\tilde{\psi} - \tilde{n}_i e^{\tilde{\psi}} + \tilde{n}_i e^{-\tilde{\psi}} + \tilde{N}_{\text{net}} = 0$ with Jacobian $L - \text{diag}(\tilde{n}+\tilde{p})$.
- **Continuity & semi-implicit SRH:** $A_n = -\nabla_h\cdot\tilde{\mathbf{J}}_n$ and $A_p = +\nabla_h\cdot\tilde{\mathbf{J}}_p$ have positive diagonals and non-positive off-diagonals. Writing $R=(np-n_i^2)/D$ moves $p/D$ (or $n/D$) onto the diagonal so both solves remain M-matrices with non-negative RHS.
- **Boltzmann-reference subtraction ($V \le 0.30\text{ V}$):** Because $A_n(\tilde{\psi})[\tilde{n}_i e^{\tilde{\psi}}] \equiv 0$ and $A_p(\tilde{\psi})[\tilde{n}_i e^{-\tilde{\psi}+V/V_T}] \equiv 0$ identically, solving for $\delta\tilde{n} = \tilde{n} - \tilde{n}_i e^{\tilde{\psi}}$ and $\delta\tilde{p} = \tilde{p} - \tilde{n}_i e^{-\tilde{\psi}+V/V_T}$ eliminates $O(1)$ cancellation in the neutral majority-carrier regions, giving machine-precision current conservation even at nanoamp/picoamp reverse currents.

## 4. Post-Processing & Visualization Helpers

Electric field components $E_x, E_y$ are computed from face-centred FVM differences and averaged to cell centres before computing $|\mathbf{E}| = \sqrt{E_x^2 + E_y^2}$. Current density is evaluated from exact Scharfetter-Gummel face fluxes and integrated over $y$ with trapezoid weights matching the FVM half-cells on the Neumann walls.

## 5. Planar equilibrium benchmarks

Compare the simulated half-density width and peak field to the classical depletion approximation. The separate Poisson–Boltzmann reference includes mobile charge near the edges of a symmetric abrupt junction at equilibrium.

In [ ]:
import numpy as np
from IPython.display import Image, display

from config import DeviceConfig, UM
from main import FIG, curvature_study, iv_study
from postprocess import (
    abrupt_depletion_width, abrupt_peak_field, built_in_potential,
    depletion_width, electric_field, symmetric_equilibrium_peak_field,
)
from solver import Device, solve_equilibrium, solve_equilibrium_1d

cfg = DeviceConfig()
dev = Device(cfg)
eq = solve_equilibrium(dev)
x_ref, psi_ref = solve_equilibrium_1d(cfg, 4 * (cfg.Nx - 1) + 1, dev.w)
psi_row = dev.reshape(eq.psi)[cfg.Ny // 2] * cfg.VT
error = np.max(np.abs(psi_row - np.interp(dev.mesh.x, x_ref, psi_ref)))
assert error < 0.01 * built_in_potential(cfg)
print(f"Built-in voltage: {built_in_potential(cfg):.6f} V")
print(f"2D vs fine 1D potential difference: {error * 1e3:.3f} mV")
print(f"Half-density width: {depletion_width(dev, eq) / UM:.4f} µm")
print(f"Classical abrupt width: {abrupt_depletion_width(cfg) / UM:.4f} µm")
print(f"Simulated peak field: {electric_field(dev, eq)[4].max() / 1e3:.3f} kV/cm")
print(f"Classical abrupt peak field: {abrupt_peak_field(cfg) / 1e3:.3f} kV/cm")
print(f"Abrupt Poisson–Boltzmann peak field: {symmetric_equilibrium_peak_field(cfg) / 1e3:.3f} kV/cm")

## 6. Planar Bias Sweep ($-3.0\text{ V} \to +0.65\text{ V}$) & Field/Carrier Dashboards

We check that $L_x \ge 3 W_d(-3.0\text{ V})$, run the full forward and reverse Gummel voltage sweep, verify FVM current conservation across all $N_x - 1$ columns, inspect the fitted forward ideality factor, and render 2×2 dashboards at equilibrium ($0\text{ V}$), forward bias ($+0.60\text{ V}$), and reverse bias ($-3.0\text{ V}$).

In [ ]:
_, states = iv_study(cfg)
for name in ("iv_planar.png", "dashboard_planar_+0.0V.png",
             "dashboard_planar_+0.6V.png", "dashboard_planar_-3.0V.png",
             "srh_generation_reverse.png"):
    display(Image(filename=str(FIG / name)))

## 7. Curvature and doping-profile comparison

Both profiles use the same cosine geometry, nominal transition scale and bias. One measures the transition horizontally and the other measures signed distance to the interface. The sweep compares both profiles across amplitudes and repeats the planar and largest-amplitude cases on the refined grid. The maps mark sampled field maxima and curvature extrema. The normal profiles use the maximum within one classical depletion width on either side of the interface.

In [ ]:
curvature_rows = curvature_study(cfg)
for name in ("emax_vs_curvature.png", "curvature_field_comparison.png",
             "curvature_normal_profiles.png"):
    display(Image(filename=str(FIG / name)))